# Week 8 — Day 1: Modal & SpecialistAgent

## The Price is Right — Capstone Project

**Plan for the week:**
- **Day 1** — Modal.com basics + SpecialistAgent (this notebook)
- **Day 2** — RAG, FrontierAgent, Ensemble Agent
- **Day 3** — ScannerAgent, MessengerAgent
- **Day 4** — AutonomousPlannerAgent and DealAgentFramework
- **Day 5** — Final autonomous deal-hunting system

Today: get familiar with Modal (serverless GPU compute), deploy our fine-tuned price predictor as a remote Modal service, and wrap it in a `SpecialistAgent`.

## Setup

In [ ]:
import os
import locale
import modal
from agents.preprocessor import Preprocessor
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
# Sanity-check encoding so special characters render correctly
print(locale.getpreferredencoding())  # should print 'UTF-8'
os.environ["PYTHONIOENCODING"] = "utf-8"

## Modal token setup

1. Sign up at https://modal.com and go to **Settings → API Tokens → New Token**
2. Run the command Modal gives you, prefixed with `uv run`:
   ```
   uv run modal token set --token-id ak-... --token-secret as-...
   ```
3. Or add directly to `.env` and rerun `load_dotenv(override=True)`:
   ```
   MODAL_TOKEN_ID=ak-...
   MODAL_TOKEN_SECRET=as-...
   ```

## First Modal app — `hello`

`local()` runs the function locally; `remote()` runs it on Modal's infrastructure.

In [ ]:
from hello import app, hello, hello_europe

In [ ]:
with app.run():
    reply = hello.local()
reply

In [ ]:
with app.run():
    reply = hello.remote()
reply

## Region-specific deployments

`@app.function(image=image, region="eu")` pins the function to a specific Modal region. See the [region selection docs](https://modal.com/docs/guide/region-selection).

In [ ]:
with app.run():
    reply = hello_europe.remote()
reply

## Set the HuggingFace token as a Modal Secret

Modal needs the HF token at runtime to pull the fine-tuned model.

1. Go to **modal.com → Secrets → Create new secret → Hugging Face**
2. **Name:** `huggingface-secret`
3. **Key:** `HF_TOKEN` &nbsp; **Value:** `hf_...`

(The pricer scripts reference `huggingface-secret` by name, so it must match exactly.)

## Run the base Llama 3.2 model on Modal

Pulls the model from HuggingFace and runs inference on a T4 GPU.

In [ ]:
from llama import app, generate

In [ ]:
with modal.enable_output():
    with app.run():
        result = generate.remote("The quick brown fox jumps over the")
result

## Run the fine-tuned price predictor as an ephemeral Modal app

In [ ]:
from pricer_ephemeral import app, price

In [ ]:
with modal.enable_output():
    with app.run():
        result = price.remote(
            "Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio"
        )
result

## Preprocessing — clean and normalize the product description before pricing

Default uses local Llama; you can switch to Groq's hosted model for speed.

In [ ]:
preprocessor = Preprocessor()
text = preprocessor.preprocess(
    "Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio"
)
print(text)

In [ ]:
preprocessor = Preprocessor(model_name="groq/openai/gpt-oss-20b")
text = preprocessor.preprocess(
    "Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio"
)
print(text)

To make Groq the default preprocessor, add this to `.env`:
```
PRICER_PREPROCESSOR_MODEL=groq/openai/gpt-oss-20b
```

In [ ]:
with modal.enable_output():
    with app.run():
        result = price.remote(text)
print(result)

In [ ]:
print(text)

## Move from ephemeral apps to a Deployed App

`uv run modal deploy <module>` packages the function as a long-running, callable Modal service. This is how you'd ship a real AI service.

**Note on secret naming:** in `pricer_service.py` and `pricer_service2.py` the secret is referenced as `huggingface-secret`. If yours is named differently in Modal (e.g. `hf-secret`), update the script to match. Visit https://modal.com/secrets/ to check.

In [ ]:
!uv run modal deploy -m pricer_service

In [ ]:
pricer = modal.Function.from_name("pricer-service", "price")

In [ ]:
# First call can be slow (cold start). Subsequent calls are fast.
pricer.remote(text)

## Deploy as a Modal Class for cached model loading

`pricer_service2.py` uses `@app.cls()` so the model is loaded once when the container starts and reused across requests — much faster than reloading on every call.

In [ ]:
!modal deploy -m pricer_service2

In [ ]:
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
reply = pricer.price.remote(text)
print(reply)

In [ ]:
# Second call should be near-instant (container already warm)
reply = pricer.price.remote(text)
print(reply)

## Optional — keep Modal containers warm

First call: ~10 minutes (image build). Subsequent: 30 sec cold, 2 sec warm.

**Option A** — set `MIN_CONTAINERS = 1` in `pricer_service2.py` (always-on, costs credits).

**Option B** — extend the warm window from 2 min → 20 min:
```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=1200)  # 20 min
```

**Revert to default:**
```python
pricer.update_autoscaler(scaledown_window=120)  # 2 min
```

## Wrap it all in our SpecialistAgent

The agent does preprocessing + remote pricing in one call. By default it preprocesses with local Llama 3.2; set `PRICER_PREPROCESSOR_MODEL=groq/openai/gpt-oss-20b` in `.env` to use Groq instead.

In [ ]:
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.specialist_agent import SpecialistAgent

agent = SpecialistAgent()

In [ ]:
agent.price("iPhone 10")